# The Cognitive of Finances
## Applying Sentiment Analysis to Time Series in Understanding Shifts in the Renewable Energy Stock Market

**Jean D. Treves** — *Master of Arts in Quantitative Methods in the Social Sciences*
*Columbia University, Graduate School of Arts and Sciences — QMSS 5999, Fall 2023*

---

### Abstract

This notebook reproduces the empirical pipeline of the thesis: a **FinBERT × SARIMAX** framework investigating whether investor sentiment derived from quarterly earnings reports significantly predicts stock price movements in renewable energy equities, and whether this relationship holds before and after the COVID-19 structural break.

### Research Question

> Does investor sentiment derived from quarterly earnings reports (10-Q) significantly predict > stock price movements in renewable energy equities — and does this relationship hold before > and after COVID-19?

The study tests whether **cognitive biases** (anchoring, loss aversion) leave measurable traces in stock prices, challenging the **Efficient Market Hypothesis (EMH)**.

### Key Finding

The sign of the sentiment-return coefficient **inverted** from **+79.55 (pre-COVID, p<0.001)** to **−90.23 (post-COVID, p=0.039)** for the renewable energy universe — a complete reversal of investor reaction to corporate communication.

---

**Companion documents:**
- Full thesis PDF: see repository
- Limitations & robustness: [`docs/LIMITATIONS.md`](../docs/LIMITATIONS.md)
- References: [`docs/references.bib`](../docs/references.bib)


## 1. Setup & Imports

Reproducibility constraints:
- Python 3.10+
- The original pipeline relied on the **Financial Modeling Prep (FMP) API**, which has since become paywalled for the `earnings-surprises` endpoint. This notebook loads **pre-computed results from the thesis** so the analysis remains accessible.
- The Google News scraper (`crawler.py`) is included for reference but **no longer functional** due to Google's anti-bot defences (2024+). FinBERT inference on cached articles remains demonstrable.


In [5]:
from __future__ import annotations

import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

# Time series & statistics
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Setup
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger(__name__)
warnings.filterwarnings("ignore", category=FutureWarning)

# Project paths
DATA_DIR: Path = Path("../data/processed")
ASSETS_DIR: Path = Path("../assets")

# Universe constants — confirmed from thesis pp.13–14
TICKERS: list[str] = ["FSLR", "GE", "NEE", "TSLA", "PLUG"]
COMPANIES: dict[str, str] = {
    "FSLR": "First Solar (Solar Power)",
    "GE":   "General Electric (Hydro / Wind)",
    "NEE":  "NextEra Energy (Wind Power)",
    "TSLA": "Tesla (Electric Vehicles)",
    "PLUG": "Plug Power (Hydrogen / Fuel Cells)",
}
COVID_BREAK_DATE: str = "2020-04-01"  # Q1/Q2 2020 boundary

logger.info("Setup complete | %d tickers | break at %s", len(TICKERS), COVID_BREAK_DATE)


2026-05-15 11:37:38,071 | INFO | Setup complete | 5 tickers | break at 2020-04-01


## 2. Load Pre-Computed Results

The empirical estimates below are extracted directly from the thesis PDF (pp. 22–69) and stored as CSVs for transparent inspection.


In [6]:
# Load all pre-computed results
stationarity_df: pd.DataFrame = pd.read_csv(DATA_DIR / "stationarity_tests.csv")
long_run_df: pd.DataFrame = pd.read_csv(DATA_DIR / "long_run_sarimax.csv")
short_run_df: pd.DataFrame = pd.read_csv(DATA_DIR / "short_run_sarimax.csv")
final_df: pd.DataFrame = pd.read_csv(DATA_DIR / "final_sarimax_pre_post_covid.csv")
sentiment_df: pd.DataFrame = pd.read_csv(DATA_DIR / "sentiment_scores_sample.csv")

# Validation
logger.info("Stationarity tests:  %s", stationarity_df.shape)
logger.info("Long-run SARIMAX:    %s", long_run_df.shape)
logger.info("Short-run SARIMAX:   %s", short_run_df.shape)
logger.info("Final pre/post COVID: %s", final_df.shape)
logger.info("Sentiment sample:    %s", sentiment_df.shape)


2026-05-15 11:37:38,097 | INFO | Stationarity tests:  (5, 8)
2026-05-15 11:37:38,098 | INFO | Long-run SARIMAX:    (5, 20)
2026-05-15 11:37:38,099 | INFO | Short-run SARIMAX:   (5, 13)
2026-05-15 11:37:38,099 | INFO | Final pre/post COVID: (2, 21)
2026-05-15 11:37:38,100 | INFO | Sentiment sample:    (20, 6)


## 3. Pipeline Architecture

The full thesis pipeline integrates three data sources and four modelling stages:

```text
Earnings Call Text
  → Web crawler (BeautifulSoup + Google Search top-10 results)
  → FinBERT-tone sentiment scoring (yiyanghkust/finbert-tone)
      · Chunked tokenization (512-token windows with CLS/SEP)
      · Weighted score: Positive (+1) / Negative (-1) / Neutral (0)
      · Output: continuous sentiment_score ∈ [-1, 1]
        ↓
Financial Data (yfinance + Financial Modeling Prep API)
  → 10-year daily OHLCV prices
  → Quarterly Surprise EPS = actual − estimated earnings
        ↓
Stationarity Tests (ADF + KPSS) per ticker → first differencing
        ↓
SARIMAX Regression (auto_arima order selection)
  → Long-run: full 2014–2023 period per ticker
  → Short-run: ±5 days window around each earnings date
  → Final model: pre-COVID (2014–Q1 2020) vs post-COVID (Q2 2020–2023)
        ↓
Diagnostics: Durbin-Watson · Shapiro-Wilk · Ljung-Box · Jarque-Bera
```

**Companies studied** (thesis pp.13–14, ranked by market cap as of 2023):

| Ticker | Company | Sector | Market Cap (2023) |
|--------|---------|--------|-------------------|
| FSLR | First Solar | Solar | $22.5B |
| GE | General Electric | Hydro / Wind | $125.5B |
| NEE | NextEra Energy | Wind | $111.6B |
| TSLA | Tesla | Electric Vehicles | $682.4B |
| PLUG | Plug Power | Hydrogen / Fuel Cells | $3.6B |


**Universe selection rationale:** GE and TSLA are included as *transition-economy proxies* — GE via its GE Vernova wind/hydro segment (>30% of revenue by 2022), TSLA as the dominant EV manufacturer whose valuation is structurally tied to renewable energy adoption narratives. Both were covered in mainstream ESG indices (MSCI ESG, S&P 500 ESG) throughout the study period, making their earnings-call sentiment directly relevant to green-transition investor flows.

## 4. Stationarity Tests (ADF + KPSS)

**Hypotheses:**
- **ADF** (Augmented Dickey-Fuller): H₀ = unit root present (non-stationary)
- **KPSS** (Kwiatkowski-Phillips-Schmidt-Shin): H₀ = series is stationary

Joint use is standard practice for time series in finance because each test has different failure modes. When ADF fails to reject H₀ **and** KPSS rejects H₀, we conclude non-stationarity requiring first differencing.


In [7]:
# Display pre-computed stationarity results (thesis Figure 3, pp.22-23)
display_cols: list[str] = ["ticker", "adf_stat", "adf_pvalue", "kpss_stat", "kpss_pvalue"]
styled = (
    stationarity_df[display_cols]
    .rename(columns={
        "adf_stat": "ADF stat",
        "adf_pvalue": "ADF p-val",
        "kpss_stat": "KPSS stat",
        "kpss_pvalue": "KPSS p-val",
    })
    .round(3)
)
logger.info("All 5 tickers: ADF fails to reject H0 AND KPSS rejects H0 → first differencing required")
styled


2026-05-15 11:37:38,111 | INFO | All 5 tickers: ADF fails to reject H0 AND KPSS rejects H0 → first differencing required


,ticker,ADF stat,ADF p-val,KPSS stat,KPSS p-val
0,GE,-1.169,0.687,0.542,0.032
1,NEE,-2.696,0.075,0.830,0.010
2,TSLA,1.681,0.998,0.687,0.015
3,PLUG,-1.937,0.315,0.471,0.048
4,FSLR,2.265,0.999,0.592,0.023


In [8]:
# Live verification on yfinance (reproducible, no FMP needed)
# Re-runs the ADF test on FSLR closing prices to demonstrate the methodology
fslr_data: pd.DataFrame = yf.Ticker("FSLR").history(start="2014-01-01", end="2024-01-01")
fslr_prices: pd.Series = fslr_data["Close"].dropna()

adf_stat, adf_pvalue, _, _, _, _ = adfuller(fslr_prices, autolag="AIC")
kpss_stat, kpss_pvalue, _, _ = kpss(fslr_prices, regression="c", nlags="auto")

logger.info("FSLR live ADF:  stat=%.3f | p-value=%.3f", adf_stat, adf_pvalue)
logger.info("FSLR live KPSS: stat=%.3f | p-value=%.3f", kpss_stat, kpss_pvalue)
logger.info("Both tests confirm non-stationarity — consistent with thesis (Q3 2023 cutoff)")


/tmp/ipykernel_81/715603648.py:7: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_pvalue, _, _ = kpss(fslr_prices, regression="c", nlags="auto")
2026-05-15 11:37:38,792 | INFO | FSLR live ADF:  stat=-0.592 | p-value=0.873
2026-05-15 11:37:38,793 | INFO | FSLR live KPSS: stat=4.461 | p-value=0.010
2026-05-15 11:37:38,794 | INFO | Both tests confirm non-stationarity — consistent with thesis (Q3 2023 cutoff)


![Stationarity tests heatmap](../assets/stationarity_tests.png)

*All 5 tickers exhibit non-stationarity (ADF) combined with rejected stationarity (KPSS), requiring first differencing before modelling.*


## 5. Long-Run SARIMAX: Surprise EPS as Predictor

For each ticker, an `auto_arima` order selection was performed (max p=3, q=3, no seasonality). The selected order for all five companies was **ARIMA(0,1,0) — a pure random walk** — consistent with weak-form EMH.

The final SARIMAX model regresses closing price on **Surprise EPS (SEPS) = actual − estimated earnings** as the exogenous variable.


In [9]:
# Display long-run SARIMAX results (thesis Figures 4-8, pp.25-37)
cols: list[str] = ["ticker", "seps_coef", "seps_pvalue", "durbin_watson_sarimax", "shapiro_wilk_p"]
long_run_display = long_run_df[cols].rename(columns={
    "seps_coef": "SEPS Coef",
    "seps_pvalue": "SEPS p-value",
    "durbin_watson_sarimax": "Durbin-Watson",
    "shapiro_wilk_p": "Shapiro-Wilk p",
}).round(4)

long_run_display


,ticker,SEPS Coef,SEPS p-value,Durbin-Watson,Shapiro-Wilk p
0,FSLR,-1.0006,0.855,1.304,0.118
1,GE,-2.7165,0.381,1.153,0.000
2,NEE,-14.3112,0.618,1.276,0.001
3,TSLA,27.8397,0.697,1.902,0.000
4,PLUG,37.1177,0.008,2.182,0.001


**Key finding (long-run):**

- **4 of 5 tickers** (FSLR, GE, NEE, TSLA): SEPS coefficient is **not statistically significant** (p > 0.38). Stock prices follow a random walk — consistent with weak-form EMH.
- **PLUG is the sole exception**: SEPS coefficient **+37.12 (p = 0.008)** — strongly significant. The thesis attributes this to PLUG operating in a *nascent hydrogen market* with lower informational efficiency than mature peers (pp.38–39).

![Long-run SARIMAX forest plot](../assets/long_run_forest.png)


## 6. FinBERT Sentiment Extraction

### Methodology

The sentiment pipeline uses **FinBERT-tone** (Huang, Wang & Yang 2023), a BERT model fine-tuned on a 4.9-billion-token corpus of financial communications:
- 2.5B tokens: 10-K and 10-Q corporate reports
- 1.3B tokens: earnings call transcripts
- 1.1B tokens: analyst reports

### Pipeline

1. For each quarter and ticker, query Google for the top-10 results matching `"<company> <quarter> <year> results"`
2. Scrape article text with BeautifulSoup
3. **Chunk tokenization** into 512-token windows (BERT's max input) with CLS/SEP tokens preserved
4. Apply FinBERT to each chunk, get `{Positive, Neutral, Negative}` probabilities
5. Weight: `score = Σ (label_weight × probability) / n_chunks` where weights are `+1 / 0 / -1`

### Operational status (2026)

> The original Google scraper (`crawler.py`) no longer works due to Google's post-2024 anti-bot > measures. The FinBERT inference logic remains valid and reproducible on cached articles.


In [10]:
# Sample of pre-computed sentiment scores (full dataset: 200 quarter-ticker pairs)
sentiment_df.head(10)


,ticker,quarter_query,sentiment_score,quarter_label,year,article_count_estimated
0,FSLR,First Solar third quarter 2023 results,0.347,Q3,2023,10
1,FSLR,First Solar second quarter 2023 results,0.412,Q2,2023,10
2,FSLR,First Solar first quarter 2023 results,0.298,Q1,2023,10
3,FSLR,First Solar fourth quarter 2022 results,0.156,Q4,2022,10
4,FSLR,First Solar first quarter 2020 results,-0.234,Q1,2020,10
5,FSLR,First Solar fourth quarter 2019 results,0.523,Q4,2019,10
6,GE,General Electric third quarter 2023 results,0.187,Q3,2023,10
7,GE,General Electric second quarter 2023 results,0.221,Q2,2023,10
8,GE,General Electric first quarter 2020 results,-0.412,Q1,2020,10
9,NEE,NextEra third quarter 2023 results,0.301,Q3,2023,10


![Sentiment distribution per ticker](../assets/sentiment_distribution.png)

### Reference implementation (FinBERT chunked inference)

The code below is **non-executed** in this notebook (model is ~1.3 GB to download). It documents the exact algorithm used in the thesis.


In [11]:
"""
Reference implementation — NOT executed in this notebook.
Documented for methodological transparency.
"""
# from transformers import BertTokenizer, BertForSequenceClassification, pipeline
#
# LABEL_WEIGHTS: dict[str, int] = {"Neutral": 0, "Positive": 1, "Negative": -1}
# MAX_CHUNK_LENGTH: int = 512
#
# def chunked_finbert_score(text: str, nlp_pipeline, tokenizer) -> float:
#     """Apply FinBERT to text in 512-token chunks, return weighted mean score in [-1, 1]."""
#     tokens: list[int] = tokenizer.encode(text, add_special_tokens=False)
#     chunks: list[list[int]] = [
#         tokens[i : i + MAX_CHUNK_LENGTH]
#         for i in range(0, len(tokens), MAX_CHUNK_LENGTH)
#     ]
#     scores: list[float] = []
#     for chunk in chunks:
#         chunk_with_special: list[int] = (
#             [tokenizer.cls_token_id] + chunk + [tokenizer.sep_token_id]
#         )
#         if len(chunk_with_special) > 512:
#             continue
#         chunk_text: str = tokenizer.decode(chunk_with_special, skip_special_tokens=True)
#         result = nlp_pipeline(chunk_text)[0]
#         scores.append(LABEL_WEIGHTS[result["label"]] * result["score"])
#     return float(np.mean(scores)) if scores else 0.0
"FinBERT pipeline shown above for reference — runnable with transformers + torch installed."


'FinBERT pipeline shown above for reference — runnable with transformers + torch installed.'

## 7. Final Model — Pre-COVID Regression

**Period:** Q1 2014 — Q1 2020 (114 pooled observations across 5 firms)

**Specification:** SARIMAX(0, 2, 1) with sentiment_score + SEPS as exogenous variables


In [12]:
# Pre-COVID results (thesis Figure 15, pp.65-67)
pre_covid: pd.Series = final_df[final_df["period"] == "pre_covid"].iloc[0]

pre_covid_summary: pd.DataFrame = pd.DataFrame({
    "Variable": ["sentiment_score", "earnings_surprise (SEPS)"],
    "Coefficient": [pre_covid["sentiment_coef"], pre_covid["seps_coef"]],
    "p-value": [pre_covid["sentiment_pvalue"], pre_covid["seps_pvalue"]],
    "Significant (α=0.05)": [
        "✅" if pre_covid["sentiment_pvalue"] < 0.05 else "❌",
        "✅" if pre_covid["seps_pvalue"] < 0.05 else "❌",
    ],
})
pre_covid_summary


,Variable,Coefficient,p-value,Significant (α=0.05)
0,sentiment_score,79.5547,0.000,✅
1,earnings_surprise (SEPS),1.0008,0.941,❌


**Pre-COVID conclusion:**

Sentiment is the **primary driver** of renewable stock prices (coefficient +79.55, p < 0.001). Surprise EPS lacks significance (p = 0.941). The thesis interprets this as evidence that the pre-pandemic renewable market was propelled by **positive cognitive beliefs around the green transition**, not earnings fundamentals (pp.65–68).


## 8. Final Model — Post-COVID Regression

**Period:** Q2 2020 — Q3 2023 (77 pooled observations across 5 firms)

**Specification:** Same SARIMAX(0, 2, 1)


In [13]:
# Post-COVID results (thesis Figure 16, pp.69-71)
post_covid: pd.Series = final_df[final_df["period"] == "post_covid"].iloc[0]

post_covid_summary: pd.DataFrame = pd.DataFrame({
    "Variable": ["sentiment_score", "earnings_surprise (SEPS)"],
    "Coefficient": [post_covid["sentiment_coef"], post_covid["seps_coef"]],
    "p-value": [post_covid["sentiment_pvalue"], post_covid["seps_pvalue"]],
    "Significant (α=0.05)": [
        "✅" if post_covid["sentiment_pvalue"] < 0.05 else "❌",
        "✅" if post_covid["seps_pvalue"] < 0.05 else "❌",
    ],
})
post_covid_summary


,Variable,Coefficient,p-value,Significant (α=0.05)
0,sentiment_score,-90.2297,0.039,✅
1,earnings_surprise (SEPS),75.3766,0.120,❌


### Hero Finding: Sign Inversion

![Sentiment-return inversion](../assets/hero_finding.png)

**Post-COVID, the sentiment coefficient inverted to −90.23 (p = 0.039)** — positive sentiment now associates with *lower* prices. The thesis offers two interpretations (pp.70–73):

1. **Structural skepticism**: Investors became sceptical of management optimism following pandemic-induced macroeconomic disruption (supply chain shocks in silicon, cancelled renewable infrastructure projects).
2. **Contrarian flows**: Informed investors began fading sell-side enthusiasm during the post-COVID inflation regime, consistent with the rise of ESG short selling documented by Bloomberg (Mookerjee & Lee, 2023).

Simultaneously, **SEPS coefficient increased to +75.38** (p = 0.120) — still not significant at 5%, but a notable improvement from p = 0.941 pre-COVID, suggesting a gradual shift toward fundamental-driven investing.


### Model Diagnostics: Pre vs Post-COVID

![Diagnostics panel](../assets/diagnostics_panel.png)


In [14]:
# Side-by-side diagnostics (thesis pp.65, 69)
diag_cols: list[str] = [
    "period", "n_obs", "ljung_box_q", "jarque_bera",
    "heteroskedasticity_h", "prob_jb", "prob_h",
]
final_df[diag_cols].rename(columns={
    "n_obs": "N",
    "ljung_box_q": "Ljung-Box Q",
    "jarque_bera": "Jarque-Bera",
    "heteroskedasticity_h": "Heterosked. H",
    "prob_jb": "JB p-val",
    "prob_h": "H p-val",
}).round(3)


,period,N,Ljung-Box Q,Jarque-Bera,Heterosked. H,JB p-val,H p-val
0,pre_covid,114,36.68,0.91,0.56,0.63,0.09
1,post_covid,77,23.53,0.07,1.44,0.97,0.38


The post-COVID model achieves **near-normal residual distribution** (JB p = 0.97) and **non-significant heteroskedasticity** (H p = 0.38), a marked improvement over the pre-COVID specification.


## 9. Conclusions

1. **All five tickers follow a random walk in the long run** — consistent with weak-form EMH (Fama 1970).
2. **Sentiment significantly drove prices pre-COVID** (coefficient +79.55, p < 0.001), consistent with cognitive biases (anchoring, loss aversion) shaping early renewable market growth.
3. **Post-COVID, sentiment inverted** (coefficient −90.23, p = 0.039): positive sentiment now associates with lower prices, reflecting market skepticism and post-pandemic uncertainty.
4. **PLUG Power deviates from EMH** with statistically significant SEPS coefficient (p = 0.008) — consistent with its nascent hydrogen market stage and lower investor coverage pre-2020.
5. **Increasing SEPS significance post-COVID** (p improved from 0.941 → 0.120) supports a long-run convergence toward market efficiency — Graham's "weighing machine" (1965).

## 10. Limitations & Robustness

This thesis is presented with explicit acknowledgement of its statistical constraints:

- **Sample size.** N=114 pooled observations pre-COVID, N=77 post-COVID are below conventional thresholds for stable SARIMAX inference (Hyndman & Athanasopoulos 2021, §9 recommend N≥200 for reliable MA estimation).
- **Multiple testing.** 5 tickers × multiple specifications without Bonferroni/FDR correction.
- **Bootstrap stability.** Coefficients warrant validation via block bootstrap (Politis & Romano 1994) before causal interpretation.
- **Structural break detection.** COVID split is exogenously imposed; Bai-Perron (2003) would identify breakpoints endogenously.
- **Crawler fragility.** Google News URLs are non-deterministic across runs; the original scraper is no longer functional due to Google's post-2024 anti-bot measures.
- **FMP API paywall.** The `earnings-surprises` endpoint became paywalled after 2023; reproducing the SEPS variable from scratch requires either a paid subscription or alternative data sources (e.g., SEC EDGAR 10-Q parsing).

See [`docs/LIMITATIONS.md`](../docs/LIMITATIONS.md) for the full discussion.

---

## References

- Huang, A. H., Wang, H., & Yang, Y. (2023). *FinBERT: A large language model for extracting information from financial text.* **Contemporary Accounting Research**, 40(2).
- Loughran, T., & McDonald, B. (2011). *When is a liability not a liability? Textual analysis, dictionaries, and 10-Ks.* **The Journal of Finance**, 66(1).
- Tetlock, P. C. (2007). *Giving content to investor sentiment: The role of media in the stock market.* **The Journal of Finance**, 62(3).
- Fama, E. F. (1970). *Efficient Capital Markets: A Review of Theory and Empirical Work.* **Journal of Finance**, 25(2).
- Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3rd ed.). OTexts.
- Graham, B. (1965). *The Intelligent Investor: A Book of Practical Counsel.* Harper & Row.

Full BibTeX in [`docs/references.bib`](../docs/references.bib).

---

**Author:** Jean Treves — M.A. Quantitative Methods in the Social Sciences, Columbia University (GPA 3.92, 2024)
[LinkedIn](https://www.linkedin.com/in/jean-treves-bbaa91257/) • [GitHub](https://github.com/Raeus1901) • jdt2175@columbia.edu
